# Clean trials from the optic flow + the log  →  `df_trials_clean.pkl`

The **second** optic-flow notebook. Notebook 0 (`single_session_optic_flow.ipynb`) produced the raw
per-ROI optic flow (`opticflow_*_*.npy`) and the pupil track. This one turns those into the **final
`df_trials_clean.pkl`** — the enriched, disjoint clean-trial table the single-session analysis uses —
so you can see the results and play with them.

The pipeline, in order (each step reuses the validated module, nothing is reimplemented here):

| step | what | module |
|---|---|---|
| **A** | one per-frame dataframe from the optic-flow `.npy` files (+ pupil) | this notebook |
| **B** | QC + smooth/fix the per-animal detectors — **iris/blink, licking, grooming** | `common/detectors/*` |
| **C** | build the clean trials from the log, then fold the optic flow in | `build_trials → enrich_trials → cluster_paths → label_trials` |
| **D** | see the results & play | this notebook |

The clean trials are **spawn-batch trials** (`trial = spawn`, disjoint reward-to-reward windows); enrich
adds the per-trial event fractions + motor/whisker features, cluster adds the Direct/Corner path group,
labels add error/conflict. This is the `banish_multiplier` task.

> **Prerequisite.** Run notebook 0 first (per-ROI flow + pupil). This notebook needs
> `opticflow/opticflow_*_*.npy`, `opticflow/pupil_track.npz`, `whisker.npz`, and `log.json`.

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

_HERE = Path.cwd()
def _up(marker, rel):
    for base in (_HERE, _HERE.parent, _HERE.parent.parent, _HERE.parent.parent.parent):
        if (base / rel / marker).exists(): return base / rel
    return None
_COMMON = _up('compute_roi_flow.py', 'common')
_TASK   = _up('build_trials.py', 'task_banish_multiplier')
assert _COMMON and _TASK, f'cannot locate common/ + task_banish_multiplier/ from {_HERE}'
sys.path.insert(0, str(_COMMON)); sys.path.insert(0, str(_COMMON / 'detectors')); sys.path.insert(0, str(_TASK))
import compute_roi_flow as rflow
import compute_eye_events, detect_licking, detect_grooming, compute_mouth_state
import build_trials, enrich_trials, cluster_paths, label_trials
print('modules loaded from', _COMMON, 'and', _TASK)

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
MAIN_DIR = '/path/to/MAIN_DIR'    # <-- server root that holds the animal folders
MOUSE_ID = 'MOUSE_ID'             # <-- animal folder name
date     = ''                     # <-- session sub-folder (recording date/time); '' = none

# Safety switches. Default: LOAD what's already there so you can look first.
REBUILD_DETECTORS = False   # True -> re-run eye/lick/groom/mouth (fast; after you change a tunable below)
REBUILD_TRIALS    = False   # True -> rebuild df_trials_clean (build -> enrich -> cluster -> labels)
SAVE_FRAMES_DF    = False   # True -> also save the per-frame table to opticflow/frames_df.pkl

SESSION_DIR = (Path(MAIN_DIR) / MOUSE_ID / date).resolve()
assert SESSION_DIR.exists(), f'session folder does not exist: {SESSION_DIR}'
sess = json.load(open(SESSION_DIR / 'session.json'))
od = SESSION_DIR / 'opticflow'
ROIS = list(sess['rois']); FPS = float(sess['fps']); N = int(sess['n_frames'])

def _ok(p): return '✓' if Path(p).exists() else '✗ MISSING'
print('session :', SESSION_DIR, '| mouse', sess.get('mouse_id', MOUSE_ID), '| task', sess.get('task_type'))
print('rois    :', ROIS)
print('flow npy:', _ok(od / f'opticflow_{ROIS[0]}_mag.npy'), '| pupil:', _ok(od / 'pupil_track.npz'),
      '| whisker:', _ok(od / 'whisker.npz'), '| log:', _ok(SESSION_DIR / 'log.json'))
print('detectors: eye', _ok(od / 'eye_events.npz'), '| lick', _ok(od / 'licking.npz'),
      '| groom', _ok(od / 'groom_mask_clean.npy'), '| mouth', _ok(od / 'mouth_state.npz'))
print('df_trials_clean:', _ok(SESSION_DIR / 'df_trials_clean.pkl'))

## A — one per-frame dataframe from the optic-flow `.npy` files

Assemble every `opticflow_<roi>_<metric>.npy` (+ the pupil radius/centre) into a single table, one row
per video frame. Columns are `<roi>_<metric>` (e.g. `paw_mag`, `whisker_right_variance`) plus
`radius`/`pupil_cx`/`pupil_cy`. This is the raw signal the detectors and the enrichment read.

In [ ]:
def build_frames_df():
    cols = {}
    for roi in ROIS:
        for m in rflow.METRICS:
            p = od / f'opticflow_{roi}_{m}.npy'
            if p.exists(): cols[f'{roi}_{m}'] = np.load(p)
    tr = np.load(od / 'pupil_track.npz', allow_pickle=True)
    for k, name in (('radius', 'radius'), ('cx', 'pupil_cx'), ('cy', 'pupil_cy')):
        if k in tr.files: cols[name] = tr[k]
    n = min(len(v) for v in cols.values())
    df = pd.DataFrame({k: v[:n] for k, v in cols.items()})
    df.index.name = 'frame'
    return df

frames = build_frames_df()
print('per-frame df:', frames.shape, '(rows = frames, cols = <roi>_<metric> + pupil)')
if SAVE_FRAMES_DF:
    frames.to_pickle(od / 'frames_df.pkl'); print('saved', od / 'frames_df.pkl')

# quick look: a few facial-motion signals + radius over a window
LO, HI = 5000, 5400
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for roi in [r for r in ('paw', 'whisker_right', 'mouth', 'nose') if f'{r}_mag' in frames]:
    ax[0].plot(frames.index[LO:HI], frames[f'{roi}_mag'][LO:HI], lw=1, label=roi)
ax[0].set_ylabel('|flow|'); ax[0].legend(fontsize=8, ncol=4); ax[0].grid(alpha=.2)
ax[0].set_title(f'{MOUSE_ID}  per-frame optic flow + pupil, frames {LO}-{HI}')
if 'radius' in frames: ax[1].plot(frames.index[LO:HI], frames['radius'][LO:HI], color='#8e44ad', lw=1.1)
ax[1].set_ylabel('pupil radius (px)'); ax[1].set_xlabel('frame'); ax[1].grid(alpha=.2)
plt.tight_layout(); plt.show()
frames.head()

## B — QC + smooth/fix the detectors (iris / licking / grooming)

Each per-animal detector has a few **module-level tunables**. To fix a detection, set the constant on
the module, flip `REBUILD_DETECTORS = True` in CONFIG, and re-run the cell — it re-writes the `.npz` and
re-plots so you can eyeball it. With `REBUILD_DETECTORS = False` it just loads what's there.

⚠️ For this mouse the pupil **SIZE** is only approximate (`radius` is kept for the report but not split
into dilation/constriction) — eye **POSITION + BLINK** are the reliable eye signals here.

In [ ]:
# ---- IRIS / BLINK ----  tunables: BRIGHT_OVER_BASE, CLOSURE_DILATE, BLINK_OVER, AXR_BLINK
# e.g. to make blink detection stricter:  compute_eye_events.BRIGHT_OVER_BASE = 14
if REBUILD_DETECTORS:
    compute_eye_events.run(str(SESSION_DIR), write=True)
ev = np.load(od / 'eye_events.npz', allow_pickle=True)
radius, blink = ev['radius'].astype(float), ev['blink']
squint = ev['squint'] if 'squint' in ev.files else np.zeros(len(radius), bool)
print(f"blink {int(blink.sum())} frames / {int(np.diff(blink.astype(int)).clip(0).sum())} events | "
      f"squint {int(squint.sum())} frames | radius median {np.nanmedian(radius):.1f}px (APPROX)")
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(frames.index[LO:HI], radius[LO:HI], color='#8e44ad', lw=1.1, label='radius (APPROX)')
ax.fill_between(frames.index[LO:HI], *ax.get_ylim(), where=blink[LO:HI], color='0.6', alpha=.5, label='blink')
ax.fill_between(frames.index[LO:HI], *ax.get_ylim(), where=squint[LO:HI], color='orange', alpha=.4, label='squint')
ax.set_title('iris radius + blink/squint'); ax.legend(fontsize=8); ax.set_xlabel('frame'); plt.show()

In [ ]:
# ---- LICKING ----  tunables: BAND (Hz), K_ENV, MIN_BOUT_S, PAD_S, MIN_LICK_S
# e.g. widen the rhythm band:  detect_licking.BAND = (4.0, 13.0)
if REBUILD_DETECTORS:
    detect_licking.run(str(SESSION_DIR), write=True)
lk = np.load(od / 'licking.npz', allow_pickle=True)
lick_bouts = lk['bout_spans'] if 'bout_spans' in lk.files else np.empty((0, 2))
lick_mask = np.zeros(N, bool)
for s, e in np.atleast_2d(lick_bouts) if len(lick_bouts) else []: lick_mask[int(s):int(e)] = True
print(f"licking: {len(lick_bouts)} bouts, {100*lick_mask.mean():.1f}% of session")
fig, ax = plt.subplots(figsize=(12, 3))
if 'mouth_y' in frames: ax.plot(frames.index[LO:HI], frames['mouth_y'][LO:HI], lw=1, color='teal', label='mouth fy')
ax.fill_between(frames.index[LO:HI], *ax.get_ylim(), where=lick_mask[LO:HI], color='teal', alpha=.25, label='lick bout')
ax.set_title('licking (mouth fy rhythm)'); ax.legend(fontsize=8); ax.set_xlabel('frame'); plt.show()

In [ ]:
# ---- GROOMING ----  tunables: PAW_PCT, WHISK_PCT, JOY_STILL, MERGE_GAP, MIN_BOUT
# e.g. require the paw more clearly raised:  detect_grooming.PAW_PCT = 80
if REBUILD_DETECTORS:
    detect_grooming.run(str(SESSION_DIR), write=True)
groom = np.load(od / 'groom_mask_clean.npy').astype(bool)
n_bouts = int(np.diff(groom.astype(int)).clip(0).sum()) + int(groom[0])
print(f"grooming: {n_bouts} bouts, {100*groom.mean():.1f}% of session")
# the joystick-stillness confound check: joy speed during grooming must be LOWER than at rest
try:
    import fps as fpsmod
    log = json.load(open(SESSION_DIR / 'log.json')); fm = fpsmod.frame_to_ms(log)
    joy = np.asarray(log['joystick_t[ms]/x/y'], float)
    jspd = np.r_[0, np.hypot(np.diff(joy[:, 1]), np.diff(joy[:, 2]))]
    jf = np.interp(fm, joy[:, 0], jspd)
    print(f"  joystick speed: grooming {jf[groom].mean():.3f}  vs  rest {jf[~groom].mean():.3f} "
          f"({'OK - lower during grooming' if jf[groom].mean() < jf[~groom].mean() else 'FLIPPED - gate broken!'})")
except Exception as e:
    print('  (joystick confound check skipped:', e, ')')
fig, ax = plt.subplots(figsize=(12, 3))
for roi in [r for r in ('paw', 'whisker_right') if f'{r}_mag' in frames]:
    ax.plot(frames.index[LO:HI], frames[f'{roi}_mag'][LO:HI], lw=1, label=roi)
ax.fill_between(frames.index[LO:HI], *ax.get_ylim(), where=groom[LO:HI], color='brown', alpha=.25, label='grooming')
ax.set_title('grooming (paw + whisker |flow|, gated on joystick stillness)'); ax.legend(fontsize=8); ax.set_xlabel('frame'); plt.show()

In [ ]:
# ---- MOUTH STATE ----  combine lick + groom into one axis (0 CLOSED / 1 LICKING / 2 GROOMING),
# grooming PRIORITISED (a raised paw sweeps the mouth ROI -> flow lick-detector false-fires).
if REBUILD_DETECTORS:
    compute_mouth_state.run(str(SESSION_DIR), write=True)
ms = np.load(od / 'mouth_state.npz', allow_pickle=True)['mouth_state']
for k, lbl in ((0, 'CLOSED'), (1, 'LICKING'), (2, 'GROOMING')):
    print(f"  {lbl:9s} {100*(ms==k).mean():5.1f}%")

## C — build the clean trials, then fold the optic flow in

`build_trials` makes the disjoint spawn-batch trials from the log; `enrich_trials` adds the per-trial
event fractions + motor/whisker features from the detectors above; `cluster_paths` adds the Direct /
Corner path group; `label_trials` adds the error / conflict labels. All four read & write
`df_trials_clean.pkl`. Set `REBUILD_TRIALS = True` to regenerate; otherwise the existing table loads.

In [ ]:
dfp = SESSION_DIR / 'df_trials_clean.pkl'
if REBUILD_TRIALS or not dfp.exists():
    t0 = time.time()
    build_trials.build(str(SESSION_DIR), write=True)                       # log -> skeleton
    enrich_trials.run(str(SESSION_DIR), write=True, verbose=False)         # + optic-flow features
    cluster_paths.run(str(SESSION_DIR), write=True, verbose=False, figures=False)   # + path cluster
    label_trials.run(str(SESSION_DIR), write=True, verbose=False, figures=False)    # + error/conflict
    print(f'rebuilt df_trials_clean in {time.time()-t0:.1f}s')
else:
    print('loading existing df_trials_clean (set REBUILD_TRIALS=True to regenerate)')
dfc = pd.read_pickle(dfp)
print('df_trials_clean:', dfc.shape)
print('columns:', list(dfc.columns))

## D — see the results & play

`dfc` is the final table. Below: outcome census, the Direct/Corner split with the held-out motor/goal
signals, and a per-trial feature view. Everything downstream (the trial report, the hypotheses, the
performance scorecard) is built from exactly this table.

In [ ]:
# per-trial licking fraction for display (enrich stores groom/whisk/blink fractions, not lick)
if 'frac_licking' not in dfc.columns:
    _ms = ms if 'ms' in dir() else np.load(od / 'mouth_state.npz', allow_pickle=True)['mouth_state']
    _lk = (_ms == 1)
    dfc['frac_licking'] = [float(_lk[int(s):int(e)].mean()) if e > s else np.nan
                           for s, e in zip(dfc['start_frame'], dfc['end_frame'])]
an = dfc[dfc['analyze']] if 'analyze' in dfc else dfc
print('outcomes:'); print(dfc['outcome'].value_counts().to_string())
print(f"\nanalyzable: {len(an)} of {len(dfc)}")
if 'cluster_name' in dfc:
    print('\npath clusters (held-out motor/goal means):')
    cols = [c for c in ('joy_fine', 'whisk_sweep', 'heading_align', 'frac_licking', 'frac_whisking',
                        'path_efficiency', 'dur_s') if c in an]
    print(an.groupby('cluster_name')[cols].mean().round(3).to_string())

# a couple of plots to play with
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if 'cluster_name' in an and 'whisk_sweep' in an:
    an.boxplot(column='whisk_sweep', by='cluster_name', ax=ax[0]); ax[0].set_title('whisker sweep by path cluster')
if 'outcome' in an and 'frac_licking' in an:
    an.boxplot(column='frac_licking', by='outcome', ax=ax[1]); ax[1].set_title('licking fraction by outcome')
    for t in ax[1].get_xticklabels(): t.set_rotation(30)
plt.suptitle(''); plt.tight_layout(); plt.show()

# per-trial feature table (edit the columns to taste)
show = [c for c in ('trial', 'outcome', 'multiplier', 'world', 'cluster_name', 'is_error', 'is_conflict',
                    'path_efficiency', 'joy_fine', 'whisk_sweep', 'heading_align', 'frac_blink',
                    'frac_licking', 'frac_grooming', 'radius_mean') if c in dfc]
print('\ndf_trials_clean ->', dfp)
dfc[show].head(20)